# 02 - Analise e limpeza inicial dos dados

Este notebook analisa `csvs/01_dataset_completo.csv`, gerado pelo `01_MegaJoin.ipynb`, e cria uma copia limpa. O arquivo bruto nao e alterado.

A limpeza desta etapa remove apenas colunas 100% nulas, colunas constantes e algumas colunas específicas. Nao preenche nulos nem remove linhas.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 40)
pd.set_option('display.width', 160)

DATA_PATH = Path('csvs/01_dataset_completo.csv')
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f'Arquivo: {DATA_PATH}')
print(f'Dimensoes: {df.shape[0]:,} linhas x {df.shape[1]:,} colunas')

## Visao geral

In [2]:
resumo = {
    'linhas': len(df),
    'colunas': df.shape[1],
    'temporadas': f"{df['race_year'].min()}-{df['race_year'].max()}" if 'race_year' in df else 'coluna ausente',
    'corridas': df['rd_race_id'].nunique() if 'rd_race_id' in df else None,
    'pilotos': df['rd_driver_id'].nunique() if 'rd_driver_id' in df else None,
    'resultados vencedores': int(df['rd_position_number'].eq(1).sum()) if 'rd_position_number' in df else None,
}
pd.Series(resumo, name='valor')

linhas                        9146
colunas                        326
temporadas               2003-2026
corridas                       436
pilotos                        117
resultados vencedores          436
Name: valor, dtype: object

## Tipos de dados

In [3]:
tipos = (
    pd.DataFrame({
        'tipo': df.dtypes.astype(str),
        'valores distintos': df.nunique(dropna=True),
    })
    .groupby('tipo')
    .agg(colunas=('tipo', 'size'), valores_distintos_mediana=('valores distintos', 'median'))
    .sort_values('colunas', ascending=False)
)
tipos

,colunas,valores_distintos_mediana
tipo,,
float64,134,12.5
str,129,30.0
int64,63,13.0


## Valores ausentes

In [4]:
nulos = df.isna().sum().rename('nulos').to_frame()
nulos['percentual'] = (nulos['nulos'] / len(df) * 100).round(2)
nulos = nulos.sort_values(['nulos', 'percentual'], ascending=False)

print(f"Colunas com ao menos um nulo: {(nulos['nulos'] > 0).sum()} de {df.shape[1]}")
print(f"Colunas totalmente vazias: {(nulos['nulos'] == len(df)).sum()}")
print(f"Linhas sem nenhum nulo: {df.notna().all(axis=1).sum():,} de {len(df):,}")
print('Colunas com mais nulos:')
display(nulos[nulos['nulos'] > 0].head(30))

colunas_vazias = nulos.index[nulos['nulos'] == len(df)].tolist()
print(f'\nColunas totalmente vazias ({len(colunas_vazias)}):')
relatorio_vazias = pd.DataFrame({
    'coluna': colunas_vazias,
    'valor': [df[coluna].iloc[0] for coluna in colunas_vazias],
})
display(relatorio_vazias)

colunas_95pct = nulos.index[nulos['percentual'] >= 95].tolist()
print(f'\nColunas com pelo menos 95% de nulos: {len(colunas_95pct)}')
print(colunas_95pct)

print('\nNulos por linha:')
display(df.isna().sum(axis=1).describe().to_frame('nulos_por_linha'))

Colunas com ao menos um nulo: 159 de 326
Colunas totalmente vazias: 51


Linhas sem nenhum nulo: 0 de 9,146
Colunas com mais nulos:


,nulos,percentual
rd_practice_time,9146,100.0
rd_practice_time_millis,9146,100.0
rd_practice_gap,9146,100.0
rd_practice_gap_millis,9146,100.0
rd_practice_interval,9146,100.0
rd_practice_interval_millis,9146,100.0
rd_practice_laps,9146,100.0
rd_qualifying_time,9146,100.0
rd_qualifying_time_millis,9146,100.0
rd_qualifying_q1,9146,100.0



Colunas totalmente vazias (51):


,coluna,valor
0,rd_practice_time,NaN
1,rd_practice_time_millis,NaN
2,rd_practice_gap,NaN
3,rd_practice_gap_millis,NaN
4,rd_practice_interval,NaN
...,...,...
46,race_warming_up_date,NaN
47,race_warming_up_time,NaN
48,sess_prequali_position_number,NaN
49,sess_prequali_position_text,NaN



Colunas com pelo menos 95% de nulos: 59
['rd_practice_time', 'rd_practice_time_millis', 'rd_practice_gap', 'rd_practice_gap_millis', 'rd_practice_interval', 'rd_practice_interval_millis', 'rd_practice_laps', 'rd_qualifying_time', 'rd_qualifying_time_millis', 'rd_qualifying_q1', 'rd_qualifying_q1_millis', 'rd_qualifying_q2', 'rd_qualifying_q2_millis', 'rd_qualifying_q3', 'rd_qualifying_q3_millis', 'rd_qualifying_gap', 'rd_qualifying_gap_millis', 'rd_qualifying_interval', 'rd_qualifying_interval_millis', 'rd_qualifying_laps', 'rd_starting_grid_position_qualification_position_number', 'rd_starting_grid_position_qualification_position_text', 'rd_starting_grid_position_grid_penalty', 'rd_starting_grid_position_grid_penalty_positions', 'rd_starting_grid_position_time', 'rd_starting_grid_position_time_millis', 'rd_fastest_lap_lap', 'rd_fastest_lap_time', 'rd_fastest_lap_time_millis', 'rd_fastest_lap_gap', 'rd_fastest_lap_gap_millis', 'rd_fastest_lap_interval', 'rd_fastest_lap_interval_millis

,nulos_por_linha
count,9146.000000
mean,90.582003
std,6.286151
min,74.000000
25%,87.000000
50%,90.000000
75%,95.000000
max,137.000000


### Decisao para as colunas 100% nulas

Vamos remover da copia limpa as colunas sem nenhum valor neste recorte. Os nulos se agrupam por estes motivos:

- `rd_practice_*`, `rd_qualifying_*`, `rd_starting_grid_*` e `rd_fastest_lap_*` pertencem a outros tipos de registro em `race_data`; a consulta principal usa `RACE_RESULT`. As sessoes selecionadas estao nas colunas `sess_*`.
- `rd_pit_stop_*` e `rd_driver_of_the_day_percentage` pertencem aos tipos `PIT_STOP` e `DRIVER_OF_THE_DAY_RESULT`, que nao entram como linhas neste join.
- `sess_prequali_*` fica vazio porque nao ha resultados de pre-classificacao a partir de 2003.
- `race_sprint_*` fica vazio depois de excluir todos os fins de semana sprint pelos campos de sprint preenchidos no banco.
- As datas de pre-classificacao, FP4, quali 1/2 e warm-up nao estao preenchidas no banco original.

## Colunas sem variacao

A contagem anterior juntava colunas vazias e colunas com um valor repetido: `nunique(dropna=False)` conta o nulo como um valor distinto. O codigo separa esses dois casos e mostra o valor das colunas constantes.

As constantes sao removidas porque nao distinguem linhas nesta amostra:

- `rd_type`: todas as linhas sao `RACE_RESULT`, por causa do filtro da consulta.
- `rd_race_shared_car`: vale `0` em todas as linhas; a amostra 2003+ nao tem carros compartilhados.
- `driver_gender`: vale `MALE` em todas as linhas; nao varia neste dataset.
- `tyremfr_best_starting_grid_position`: vale `1` em todas as linhas e nao separa os resultados.
- `tyremfr_best_race_result`: vale `1` em todas as linhas e nao separa os resultados.

O codigo tambem procura strings vazias e marcadores de ausencia, sem alterar dados.

In [5]:
valores_distintos = df.nunique(dropna=False)
colunas_vazias = df.columns[df.isna().all()].tolist()
colunas_constantes = [
    coluna for coluna in df.columns
    if valores_distintos[coluna] <= 1 and coluna not in colunas_vazias
]

print(f'Colunas totalmente vazias: {len(colunas_vazias)}')
print(f'Colunas constantes com valor: {len(colunas_constantes)}')

relatorio_constantes = pd.DataFrame({
    'valor unico': [df[coluna].iloc[0] for coluna in colunas_constantes],
    'tipo': [str(df[coluna].dtype) for coluna in colunas_constantes],
}, index=colunas_constantes)
display(relatorio_constantes)

colunas_texto = [
    coluna for coluna in df.columns
    if pd.api.types.is_string_dtype(df[coluna].dtype) or df[coluna].dtype == object
]
texto_vazio = {}
marcadores = {'NULL', 'N/A', 'NA', 'NAN', r'\N'}
marcadores_encontrados = {}
for coluna in colunas_texto:
    serie = df[coluna].dropna().astype('string').str.strip()
    vazios = int(serie.eq('').sum())
    if vazios:
        texto_vazio[coluna] = vazios
    contagem = serie[serie.str.upper().isin(marcadores)].value_counts()
    if not contagem.empty:
        marcadores_encontrados[coluna] = contagem.to_dict()

print(f'Colunas de texto com strings vazias: {len(texto_vazio)}')
print(texto_vazio)
print(f'Colunas de texto com marcadores de ausencia: {len(marcadores_encontrados)}')
print(marcadores_encontrados)

Colunas totalmente vazias: 51
Colunas constantes com valor: 5


,valor unico,tipo
rd_type,RACE_RESULT,str
rd_race_shared_car,0,int64
driver_gender,MALE,str
tyremfr_best_starting_grid_position,1,int64
tyremfr_best_race_result,1,int64


Colunas de texto com strings vazias: 0
{}
Colunas de texto com marcadores de ausencia: 0
{}


### Sessoes antigas e dados biograficos

Tambem vamos retirar estas colunas da copia limpa:

- `sess_warmup_*` so tem dados em 2003; `sess_fp4_*` e `sess_quali1_*`/`sess_quali2_*` so tem dados entre 2003 e 2005. Esses formatos nao aparecem no periodo recente usado para validacao e teste.
- `driver_date_of_death` e `driver_second_nationality_country_id` foram removidas por decisao de escopo. A data de falecimento e posterior a parte dos resultados; a segunda nacionalidade so esta preenchida para tres pilotos.

O dataset bruto permanece preservado.

## Limpeza das colunas

A copia limpa remove colunas 100% nulas, constantes, sessoes antigas, campos biograficos fora do escopo, horario previsto da corrida, penalidades pos-corrida e voltas/distancia programadas. Mantem todas as linhas e os valores das demais colunas. Nao preenche nulos.

### Penalidades da corrida: removidas

`rd_race_time_penalty` e `rd_race_time_penalty_millis` so sao conhecidas durante ou depois da prova. Como o objetivo e prever o vencedor antes da largada, elas nao entram no modelo. Portanto, saem agora de `dataset_limpo`; ficam apenas no dataset bruto `dataset_completo`.

### Tempo final da corrida: removido

`rd_race_time` e `rd_race_time_millis` sao resultados da propria prova. Em 4.631 das 9.146 linhas (50,63%), nao ha tempo em milissegundos: a diferenca pode estar registrada em voltas (`+1 lap`, por exemplo), ou o piloto pode ter abandonado ou sido desclassificado. Esses casos nao permitem recuperar o tempo exato; preencher os nulos nao acrescentaria informacao confiavel. Os campos saem de `dataset_limpo` e continuam no arquivo bruto.

O historico de tempos de um piloto numa pista pode ajudar na previsao se vier apenas de corridas anteriores. O tempo da corrida que queremos prever seria leakage. Essa feature historica pode ser criada depois, sem usar o tempo do proprio evento.

### Distancia e voltas programadas: removidas

`race_scheduled_distance` e `race_scheduled_laps` estao preenchidas em apenas 35 corridas (724 linhas); 8.422 de 9.146 linhas nao tem valor. O dataset ja registra `race_distance` e `race_laps` para todas as linhas. Os campos `race_scheduled_*` descrevem excecoes ao padrao de 305 km + 1 volta, entao nao precisamos deles na copia limpa.

### Datas e horarios de treinos e classificacao: removidos

Removemos as datas e os horarios de FP1, FP2, FP3 e classificacao. Sao oito colunas, mas apenas dois tipos de informacao por sessao: data e horario. Os oito campos estao preenchidos exatamente nas mesmas 895 linhas (9,79%), correspondentes a 44 corridas de 2024 a 2026; faltam em 8.251 linhas (90,21%).

A ausencia vem da propria `formula.db`: esses valores nao estao registrados na fonte para as demais corridas. Isso nao significa que as sessoes nao aconteceram nem que nao tinham horario; ha cronogramas historicos publicados fora desta base. Nao vamos inventar nem preencher esses valores. Como a cobertura e pequena, os campos saem de `dataset_limpo` e ficam preservados em `dataset_completo` para uma eventual recuperacao ou comparacao futura. Esses horarios nao sao leakage por si so, mas ficam fora da primeira versao das features por cobertura insuficiente.

### Pais e continente do Grande Premio: removidos

`gp_country_name`, `gp_country_demonym`, `gp_country_ioc_code`, `gp_country_alpha2_code`, `gp_country_alpha3_code` e `gp_country_id.1` sao outras formas de identificar o pais do GP; `gp_continent_id`, `gp_country_continent_id`, `gp_continent_name` e `gp_continent_demonym` identificam ou nomeiam o continente. Esses campos nao acrescentam uma localizacao distinta. `gp_country_id` coincide com `circuit_country_id` em 99,1% das linhas em que ambos estao preenchidos; o pais, o local e as coordenadas do circuito permanecem na copia limpa. Os campos de pais e continente do GP ficam nulos juntos em 238 linhas (2,60%), incluindo corridas chamadas Grande Premio da Europa, sem um pais associado na fonte. Removemos essas variantes de pais e continente da copia para o modelo; `dataset_completo` preserva os campos originais.

### Codigos de continente: removidos

`gp_continent_code`, `birth_continent_code`, `nat_continent_code` e `constructor_continent_code` sao codigos textuais derivados do continente. Os respectivos paises e campos de continente por ID ou nome ja estao disponiveis; portanto, esses codigos nao acrescentam uma caracteristica geografica distinta. `gp_continent_code` ainda tem 1.272 nulos (13,91%), enquanto a localizacao do circuito esta preenchida. Removemos os quatro da copia limpa.

### Campos auxiliares de posicao e volta rapida: removidos

`dstand_position_text` repete `dstand_position_number` em todas as linhas preenchidas. `dstand_position_display_order` e `cstand_position_display_order` tambem coincidem com a posicao numerica em quase todos os casos; nas poucas diferencas, representam a ordem de exibicao, nao uma medida adicional de desempenho. Mantemos `dstand_position_number` e `cstand_position_number` como posicoes da classificacao.

`sess_fastest_lap_lap` indica em qual volta ocorreu a volta mais rapida. Para criar historico de ritmo, o tempo (`sess_fastest_lap_time_millis`) e a medida mais diretamente util; o numero da volta nao acrescenta uma medida comparavel de velocidade entre corridas. Removemos esse campo e mantemos o tempo da volta rapida.

### Posicoes textuais dos treinos livres: removidas

`sess_fp1_position_text`, `sess_fp2_position_text` e `sess_fp3_position_text` repetem exatamente, nas linhas preenchidas, os valores numericos de `sess_fp1_position_number`, `sess_fp2_position_number` e `sess_fp3_position_number`. Mantemos as colunas numericas, que representam a mesma posicao sem duplicar a informacao.

### Titulos descritivos e geografia: removidos

`race_official_name`, `gp_name`, `gp_full_name`, `gp_short_name` e `gp_abbreviation` sao titulos ou formas de exibicao do evento; os identificadores de corrida e GP permanecem para organizar os dados. `circuit_full_name` e `circuit_place_name` sao descricoes redundantes; mantemos `circuit_name` e os identificadores do circuito.

Para a V1, removemos os campos de pais, nacionalidade, local de nascimento e continente de circuito, piloto, construtor, fabricante de motor e fabricante de pneus. Esses metadados geograficos ficam fora do conjunto de features escolhido. Os identificadores de piloto, construtor, circuito, motor e pneu permanecem disponiveis para relacionar o historico de desempenho.

### Variantes de nomes e numero do piloto: removidos

`driver_first_name`, `driver_last_name`, `driver_full_name` e `driver_abbreviation` sao variantes de apresentacao do piloto; `driver_id` identifica a entidade e `driver_name` permanece como nome legivel. `driver_place_of_birth` e um local descritivo, fora do escopo geografico da V1. `constructor_full_name`, `chassis_full_names` e `engine_full_names` sao descricoes extensas; os identificadores e nomes curtos correspondentes permanecem. `rd_driver_number` e um numero de identificacao, sem medida de desempenho, e `rd_driver_id` identifica o piloto. Removemos esses campos da copia limpa; os dados originais seguem em `dataset_completo`.

### Colunas repetidas: remocao apos validacao

Os pares de identificadores e medidas listados na etapa de limpeza sao comparados linha a linha, incluindo a posicao dos nulos. Uma coluna candidata so e removida quando a funcao `colunas_iguais` confirma igualdade em 100% dos valores e tipos. Se houver qualquer diferenca ou uma das colunas nao existir, a candidata permanece.

### Coordenadas do circuito: removidas

`circuit_latitude` e `circuit_longitude` identificam a localizacao geografica exata do circuito. Para esta versao, mantemos a identificacao do circuito e o pais, sem usar as coordenadas como features. Removemos os dois campos de `dataset_limpo`; seus valores continuam preservados em `dataset_completo`. Coordenadas extremas representam locais reais e nao devem ser tratadas como erros apenas por estarem distantes.

### Metadados descritivos: removidos

`circuit_previous_names` guarda nomes antigos de circuitos; e informacao historica de identificacao, redundante com o circuito atual, sem sinal direto para a previsao. `driver_permanent_number` e um numero de identificacao do piloto, nao uma medida de desempenho; aparece em apenas 28 valores distintos e esta vazio em 5.537 linhas. Como `driver_id` ja identifica o piloto, manter o numero pode apenas facilitar que o modelo memorize essa identidade. Removemos as duas colunas de `dataset_limpo`; elas continuam no arquivo bruto.

### Tempo de classificacao: manter a coluna original

`sess_quali_time_millis` registra o tempo unico usado ate 2005. A partir de 2006, a fonte guarda os tempos separadamente em `sess_quali_q1_millis`, `sess_quali_q2_millis` e `sess_quali_q3_millis`. Nao preencheremos a coluna original com esses valores. Na engenharia de features, podemos criar uma nova medida escolhendo o ultimo segmento disponivel (Q3, senao Q2, senao Q1) e usando o tempo antigo como fallback. A medida deve incluir um indicador do segmento, pois tempos de fases diferentes nao sao diretamente comparaveis. Essa regra daria cobertura a 8.987 das 9.146 linhas (98,26%).

### Motivo de abandono: removido

`rd_race_reason_retired` descreve um motivo registrado durante ou depois da corrida. Como nao esta disponivel antes da largada, nao serve como feature para prever o resultado da mesma prova. A coluna sai de `dataset_limpo`; o valor original permanece em `dataset_completo`.

### Gaps e intervalos da corrida: removidos

`rd_race_gap`, `rd_race_gap_millis`, `rd_race_gap_laps`, `rd_race_interval` e `rd_race_interval_millis` descrevem a diferenca entre pilotos no resultado da propria corrida. Sao dados posteriores a largada e nao podem ser features pre-corrida. Os nulos de `rd_race_gap_laps` sao estruturais: em muitos casos o resultado mostra a diferenca em tempo; em outros, nao registra gap. Como essas colunas saem do dataset do modelo, nao vamos preencher esses nulos. Os originais ficam em `dataset_completo`.


In [6]:
colunas_nulas = df.columns[df.isna().all()].tolist()
cardinalidade = df.nunique(dropna=False)
colunas_constantes = [
    coluna for coluna in df.columns
    if cardinalidade[coluna] == 1 and coluna not in colunas_nulas
]
colunas_sessoes_antigas = [
    'sess_warmup_position_number', 'sess_warmup_position_text',
    'sess_warmup_time_millis', 'sess_warmup_laps',
    'sess_fp4_position_number', 'sess_fp4_position_text',
    'sess_fp4_time_millis', 'sess_fp4_laps',
    'sess_quali1_position_number', 'sess_quali1_position_text',
    'sess_quali1_time_millis', 'sess_quali1_laps',
    'sess_quali2_position_number', 'sess_quali2_position_text',
    'sess_quali2_time_millis', 'sess_quali2_laps',
]
colunas_biograficas = [
    'driver_date_of_death',
    'driver_second_nationality_country_id',
]
colunas_metadados_descritivos = [
    'circuit_previous_names',
    'driver_permanent_number',
]
colunas_coordenadas_circuito = [
    'circuit_latitude',
    'circuit_longitude',
]
colunas_variantes_identificacao = [
    'driver_first_name',
    'driver_last_name',
    'driver_full_name',
    'driver_abbreviation',
    'driver_place_of_birth',
    'constructor_full_name',
    'chassis_full_names',
    'engine_full_names',
    'rd_driver_number',
]
colunas_rotulos_evento_circuito = [
    'race_official_name',
    'gp_name',
    'gp_full_name',
    'gp_short_name',
    'gp_abbreviation',
    'circuit_full_name',
    'circuit_place_name',
]
colunas_geografia_v1 = [
    'circuit_country_id',
    'driver_country_of_birth_country_id',
    'driver_nationality_country_id',
    'nat_country_id',
    'nat_country_alpha2_code',
    'nat_country_alpha3_code',
    'nat_country_ioc_code',
    'nat_country_name',
    'nat_country_demonym',
    'nat_country_continent_id',
    'nat_continent_id',
    'nat_continent_name',
    'nat_continent_demonym',
    'birth_country_id',
    'birth_country_alpha2_code',
    'birth_country_alpha3_code',
    'birth_country_ioc_code',
    'birth_country_name',
    'birth_country_demonym',
    'birth_country_continent_id',
    'birth_continent_id',
    'birth_continent_name',
    'birth_continent_demonym',
    'constructor_country_id',
    'constructor_country_id.1',
    'constructor_country_alpha2_code',
    'constructor_country_alpha3_code',
    'constructor_country_ioc_code',
    'constructor_country_name',
    'constructor_country_demonym',
    'constructor_country_continent_id',
    'constructor_continent_id',
    'constructor_continent_name',
    'constructor_continent_demonym',
    'engmfr_country_id',
    'tyremfr_country_id',
]
colunas_codigos_continente = [
    'gp_continent_code',
    'birth_continent_code',
    'nat_continent_code',
    'constructor_continent_code',
]
colunas_posicao_texto_treinos = [
    'sess_fp1_position_text',
    'sess_fp2_position_text',
    'sess_fp3_position_text',
]
colunas_campos_auxiliares = [
    'dstand_position_text',
    'dstand_position_display_order',
    'cstand_position_display_order',
    'sess_fastest_lap_lap',
]
colunas_geografia_gp = [
    'gp_country_id',
    'gp_country_demonym',
    'gp_country_continent_id',
    'gp_continent_name',
    'gp_continent_id',
    'gp_country_ioc_code',
    'gp_country_alpha2_code',
    'gp_country_id.1',
    'gp_country_name',
    'gp_continent_demonym',
    'gp_country_alpha3_code',
]
colunas_horario = ['race_time']
colunas_calendario_sessoes = [
    'race_free_practice_1_date', 'race_free_practice_1_time',
    'race_free_practice_2_date', 'race_free_practice_2_time',
    'race_free_practice_3_date', 'race_free_practice_3_time',
    'race_qualifying_date', 'race_qualifying_time',
]
colunas_penalidade = [
    'rd_race_time_penalty',
    'rd_race_time_penalty_millis',
]
colunas_tempo_final = [
    'rd_race_time',
    'rd_race_time_millis',
]
colunas_motivo_pos_prova = ['rd_race_reason_retired']
colunas_gaps_pos_prova = [
    'rd_race_gap',
    'rd_race_gap_millis',
    'rd_race_gap_laps',
    'rd_race_interval',
    'rd_race_interval_millis',
]
colunas_programadas = [
    'race_scheduled_distance',
    'race_scheduled_laps',
]
colunas_sessoes_antigas = [c for c in colunas_sessoes_antigas if c in df.columns]
colunas_biograficas = [c for c in colunas_biograficas if c in df.columns]
colunas_metadados_descritivos = [c for c in colunas_metadados_descritivos if c in df.columns]
colunas_coordenadas_circuito = [c for c in colunas_coordenadas_circuito if c in df.columns]
colunas_rotulos_evento_circuito = [c for c in colunas_rotulos_evento_circuito if c in df.columns]
colunas_variantes_identificacao = [c for c in colunas_variantes_identificacao if c in df.columns]
colunas_geografia_v1 = [c for c in colunas_geografia_v1 if c in df.columns]
colunas_geografia_gp = [c for c in colunas_geografia_gp if c in df.columns]
colunas_codigos_continente = [c for c in colunas_codigos_continente if c in df.columns]
colunas_posicao_texto_treinos = [c for c in colunas_posicao_texto_treinos if c in df.columns]
colunas_campos_auxiliares = [c for c in colunas_campos_auxiliares if c in df.columns]
colunas_horario = [c for c in colunas_horario if c in df.columns]
colunas_calendario_sessoes = [c for c in colunas_calendario_sessoes if c in df.columns]
colunas_penalidade = [c for c in colunas_penalidade if c in df.columns]
colunas_tempo_final = [c for c in colunas_tempo_final if c in df.columns]
colunas_motivo_pos_prova = [c for c in colunas_motivo_pos_prova if c in df.columns]
colunas_gaps_pos_prova = [c for c in colunas_gaps_pos_prova if c in df.columns]
colunas_programadas = [c for c in colunas_programadas if c in df.columns]

pares_duplicados = [
    ('race_id', 'rd_race_id'),
    ('driver_id', 'rd_driver_id'),
    ('constructor_id', 'rd_constructor_id'),
    ('constructor_id', 'chassis_constructor_ids'),
    ('circuit_id', 'race_circuit_id'),
    ('circuit_id', 'layout_circuit_id'),
    ('layout_id', 'race_circuit_layout_id'),
    ('gp_id', 'race_grand_prix_id'),
    ('circuit_type', 'race_circuit_type'),
    ('race_direction', 'circuit_direction'),
    ('race_turns', 'layout_turns'),
    ('engmfr_id', 'rd_engine_manufacturer_id'),
    ('tyremfr_id', 'rd_tyre_manufacturer_id'),
    ('driver_total_points', 'driver_total_championship_points'),
    ('sess_quali_position_number', 'rd_race_qualification_position_number'),
    ('sess_quali_position_text', 'rd_race_qualification_position_text'),
]

def colunas_iguais(tabela, coluna_a, coluna_b):
    """Retorna True quando duas colunas tem mesmos valores, ordem, nulos e tipo."""
    if coluna_a not in tabela.columns or coluna_b not in tabela.columns:
        return False
    return tabela[coluna_a].equals(tabela[coluna_b])

resultado_validacao = []
for coluna_mantida, candidata in pares_duplicados:
    existem = coluna_mantida in df.columns and candidata in df.columns
    iguais = colunas_iguais(df, coluna_mantida, candidata)
    if existem:
        a, b = df[coluna_mantida], df[candidata]
        iguais_por_linha = (a.eq(b) | (a.isna() & b.isna())).fillna(False)
        linhas_iguais = int(iguais_por_linha.sum())
        linhas_comparadas = len(df)
    else:
        linhas_iguais = 0
        linhas_comparadas = len(df)
    resultado_validacao.append({
        'manter': coluna_mantida,
        'candidata': candidata,
        'iguais_100pct': iguais,
        'linhas_iguais': linhas_iguais,
        'linhas_comparadas': linhas_comparadas,
        'acao': 'remover candidata' if iguais else 'manter ambas',
    })

relatorio_duplicatas = pd.DataFrame(resultado_validacao)
colunas_duplicadas_confirmadas = relatorio_duplicatas.loc[
    relatorio_duplicatas['iguais_100pct'], 'candidata'
].tolist()
print('Validacao das colunas candidatas a duplicata:')
display(relatorio_duplicatas)
colunas_removidas = list(dict.fromkeys(
    colunas_nulas + colunas_constantes + colunas_sessoes_antigas + colunas_biograficas + colunas_metadados_descritivos + colunas_coordenadas_circuito + colunas_rotulos_evento_circuito + colunas_variantes_identificacao + colunas_geografia_v1 + colunas_geografia_gp + colunas_codigos_continente + colunas_posicao_texto_treinos + colunas_campos_auxiliares + colunas_duplicadas_confirmadas
    + colunas_horario + colunas_calendario_sessoes + colunas_penalidade + colunas_tempo_final + colunas_motivo_pos_prova + colunas_gaps_pos_prova + colunas_programadas
))

df_limpo = df.drop(columns=colunas_removidas).copy()

print(f'Colunas removidas por estarem 100% nulas: {len(colunas_nulas)}')
print(f'Colunas removidas por serem constantes: {len(colunas_constantes)}')
print(f'Colunas de sessoes antigas removidas: {len(colunas_sessoes_antigas)}')
print(f'Colunas biograficas removidas: {len(colunas_biograficas)}')
print(f'Colunas de metadados descritivos removidas: {len(colunas_metadados_descritivos)}')
print(f'Coordenadas do circuito removidas: {len(colunas_coordenadas_circuito)}')
print(f'Rotulos descritivos de evento/circuito removidos: {len(colunas_rotulos_evento_circuito)}')
print(f'Variantes de identificacao removidas: {len(colunas_variantes_identificacao)}')
print(f'Campos geograficos removidos para a V1: {len(colunas_geografia_v1)}')
print(f'Colunas geograficas do GP removidas: {len(colunas_geografia_gp)}')
print(f'Codigos de continente removidos: {len(colunas_codigos_continente)}')
print(f'Posicoes textuais de treinos removidas: {len(colunas_posicao_texto_treinos)}')
print(f'Campos auxiliares de posicao/volta rapida removidos: {len(colunas_campos_auxiliares)}')
print(f'Colunas duplicadas removidas apos validacao: {len(colunas_duplicadas_confirmadas)}')
print(f'Colunas de horario removidas: {len(colunas_horario)}')
print(f'Colunas de calendario de sessoes removidas: {len(colunas_calendario_sessoes)}')
print(f'Colunas de penalidade removidas: {len(colunas_penalidade)}')
print(f'Colunas de tempo final removidas: {len(colunas_tempo_final)}')
print(f'Colunas de motivo pos-prova removidas: {len(colunas_motivo_pos_prova)}')
print(f'Colunas de gap/intervalo pos-prova removidas: {len(colunas_gaps_pos_prova)}')
print(f'Colunas programadas removidas: {len(colunas_programadas)}')
print(f'Colunas removidas no total: {len(colunas_removidas)}')
print(f'Dimensoes antes: {df.shape[0]:,} linhas x {df.shape[1]:,} colunas')
print(f'Dimensoes depois: {df_limpo.shape[0]:,} linhas x {df_limpo.shape[1]:,} colunas')
print(f'Colunas totalmente nulas restantes: {int(df_limpo.isna().all().sum())}')
print(f'Colunas constantes restantes: {int((df_limpo.nunique(dropna=False) <= 1).sum())}')

csv_limpo = Path('csvs/02_dataset_limpo.csv')
pkl_limpo = Path('dataset_limpo.pkl')
df_limpo.to_csv(csv_limpo, index=False)
df_limpo.to_pickle(pkl_limpo)
print(f'\nArquivos salvos: {csv_limpo} e {pkl_limpo}')

Validacao das colunas candidatas a duplicata:


,manter,candidata,iguais_100pct,linhas_iguais,linhas_comparadas,acao
0,race_id,rd_race_id,True,9146,9146,remover candidata
1,driver_id,rd_driver_id,True,9146,9146,remover candidata
2,constructor_id,rd_constructor_id,True,9146,9146,remover candidata
3,constructor_id,chassis_constructor_ids,True,9146,9146,remover candidata
4,circuit_id,race_circuit_id,True,9146,9146,remover candidata
5,circuit_id,layout_circuit_id,True,9146,9146,remover candidata
6,layout_id,race_circuit_layout_id,True,9146,9146,remover candidata
7,gp_id,race_grand_prix_id,True,9146,9146,remover candidata
8,circuit_type,race_circuit_type,True,9146,9146,remover candidata
9,race_direction,circuit_direction,True,9146,9146,remover candidata


Colunas removidas por estarem 100% nulas: 51
Colunas removidas por serem constantes: 5
Colunas de sessoes antigas removidas: 16
Colunas biograficas removidas: 2
Colunas de metadados descritivos removidas: 2
Coordenadas do circuito removidas: 2
Rotulos descritivos de evento/circuito removidos: 7
Variantes de identificacao removidas: 9
Campos geograficos removidos para a V1: 36
Colunas geograficas do GP removidas: 11
Codigos de continente removidos: 4
Posicoes textuais de treinos removidas: 3
Campos auxiliares de posicao/volta rapida removidos: 4
Colunas duplicadas removidas apos validacao: 16
Colunas de horario removidas: 1
Colunas de calendario de sessoes removidas: 8
Colunas de penalidade removidas: 2
Colunas de tempo final removidas: 2
Colunas de motivo pos-prova removidas: 1
Colunas de gap/intervalo pos-prova removidas: 5
Colunas programadas removidas: 2
Colunas removidas no total: 189
Dimensoes antes: 9,146 linhas x 326 colunas
Dimensoes depois: 9,146 linhas x 137 colunas


Colunas totalmente nulas restantes: 0
Colunas constantes restantes: 0



Arquivos salvos: dataset_limpo.csv e dataset_limpo.pkl


## Nova analise de nulos apos a limpeza

Agora que removemos as colunas 100% nulas e as constantes, refazemos a analise em `df_limpo`. Esta etapa mostra os nulos parciais que continuam no dataset e prepara a proxima decisao de limpeza.

In [7]:
nulos_apos_limpeza = df_limpo.isna().sum()
colunas_com_nulos = nulos_apos_limpeza[nulos_apos_limpeza > 0].sort_values(ascending=False)
relatorio_nulos_apos_limpeza = colunas_com_nulos.rename('nulos').to_frame()
relatorio_nulos_apos_limpeza['percentual'] = (
    relatorio_nulos_apos_limpeza['nulos'] / len(df_limpo) * 100
).round(2)

print(f'Linhas: {len(df_limpo):,} | colunas: {df_limpo.shape[1]:,}')
print(f'Colunas com ao menos um nulo: {len(colunas_com_nulos)} de {df_limpo.shape[1]}')
print(f'Colunas 100% nulas: {int(df_limpo.isna().all().sum())}')
print(f'Total de valores nulos restantes: {int(nulos_apos_limpeza.sum()):,}')
print(f'Linhas sem nulos: {int(df_limpo.notna().all(axis=1).sum()):,} de {len(df_limpo):,}')
print('Colunas com mais nulos:')
display(relatorio_nulos_apos_limpeza.head(30))

Linhas: 9,146 | colunas: 137
Colunas com ao menos um nulo: 43 de 137
Colunas 100% nulas: 0
Total de valores nulos restantes: 38,972
Linhas sem nulos: 0 de 9,146
Colunas com mais nulos:


,nulos,percentual
sess_quali_time_millis,8144,89.04
sess_quali_q3_millis,5436,59.44
rd_race_driver_of_the_day,5149,56.30
rd_race_points,5034,55.04
sess_quali_q2_millis,3283,35.90
rd_race_positions_gained,1711,18.71
rd_position_number,1711,18.71
sess_quali_q1_millis,1163,12.72
sess_fp1_time_millis,895,9.79
dstand_positions_gained,575,6.29
